# Travail fait par:

-LABRAHMI Doha

-LAALJ Marwa


## Partie 1 — Recherche et téléchargement du dataset

| Question | Réponse |
|---|---|
| **Source** | Fonds monétaire international (FMI), portail *IMF Data* (<https://www.imf.org/en/Data>), tableau « *Past IMF Disbursements and Repayments for all members* » (fichier `extrep1`), téléchargé au format Excel. |
| **Domaine étudié** | Finance internationale : prêts accordés par le FMI à l'ensemble de ses pays membres et remboursements reçus, par année. |
| **Observations / variables** | **43 observations** (une par année, 1984 → 2026) et **13 variables** (l'année + 12 montants en DTS). Une 14ᵉ variable catégorielle (`decennie`) sera créée. |
| **Unité** | Droits de tirage spéciaux (**DTS**), valeurs nominales. L'année 2026 est **partielle** (jusqu'au 30 septembre 2026). |

**Les 12 montants** sont organisés en 3 « guichets » du FMI × 3 flux :
- **GRA** (*General Resources Account*, prêts aux conditions du marché) : décaissements, rachats (*repurchases*), frais payés (*charges*) ;
- **PRGT** (*Poverty Reduction and Growth Trust*, prêts concessionnels aux pays à faible revenu) : décaissements, remboursements, intérêts ;
- **RST** (*Resilience and Sustainability Trust*, guichet résilience/climat, récent) : décaissements, remboursements, intérêts ;
- **Total** : décaissements, remboursements, frais et intérêts.

**Justification du choix.** Le sujet relève de l'économie et de la finance, domaine de notre formation. Les données sont publiques, au format Excel, et permettent des agrégations SQL riches (somme, moyenne, regroupement par période, classement, parts relatives).
*Écart avec les critères du TP :* le dataset contient 43 observations (< 100). C'est justifié : c'est une **série temporelle annuelle agrégée sur tous les pays membres** ; elle est courte par nature mais couvre 40 ans de crises financières, et ses 13 variables permettent des analyses variées.

**Analyses envisagées :** années de pic de décaissements, flux net (décaissements − remboursements) par décennie, poids des prêts concessionnels (PRGT + RST), poids du coût du crédit (frais et intérêts).

## Partie 2 — Stockage sur Google Drive

**Actions manuelles à faire dans Drive** (à ne pas oublier avant d'exécuter la cellule suivante) :
1. Créer le dossier `TP_SQL_Dataset` dans *Mon Drive*.
2. Y déposer le fichier `extrep1.xls.xlsx`.
3. Ouvrir ce notebook dans Colab et exécuter la cellule ci-dessous (autoriser l'accès à Drive).

La cellule détecte automatiquement si elle tourne dans Colab (Drive monté) ou en local (dossier courant).

In [6]:
import os
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DOSSIER = Path('/content/drive/MyDrive/TP_SQL_Dataset')
    print('Environnement : Google Colab (Drive monté)')
except ImportError:
    DOSSIER = Path('.').resolve()
    print('Environnement : hors Colab -> dossier local utilisé :', DOSSIER)

FICHIER = DOSSIER / 'extrep1.xls.xlsx'
if not FICHIER.is_file():
    raise FileNotFoundError(f'Fichier introuvable : {FICHIER}')
print('✅ Fichier trouvé :', FICHIER.name, '-', FICHIER.stat().st_size, 'octets')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environnement : Google Colab (Drive monté)


FileNotFoundError: Fichier introuvable : /content/drive/MyDrive/TP_SQL_Dataset/extrep1.xls.xlsx

## Partie 3 — Exploration et préparation des données

### 3.1 Lecture brute : le fichier n'est pas « propre »
On lit d'abord le fichier **sans en-tête** pour voir sa vraie structure.

In [ ]:
brut = pd.read_excel(FICHIER, header=None)
display(brut.head(9))
print('Dimensions brutes :', brut.shape)

**Constats :** (1) 3 lignes de titre en haut, (2) un **en-tête sur deux niveaux** (guichet puis type de flux, lignes 5-6), (3) les montants sont lus comme du **texte avec virgules** (`'7,357,252,000'`). Il faut donc sauter les 6 premières lignes, donner nos propres noms de colonnes et convertir en nombres.

### 3.2 Nettoyage : noms de colonnes et conversion numérique

In [ ]:
colonnes = [
    'annee',
    'gra_decaissements', 'gra_rachats', 'gra_frais_payes',
    'prgt_decaissements', 'prgt_remboursements', 'prgt_interets_payes',
    'rst_decaissements', 'rst_remboursements', 'rst_interets_payes',
    'total_decaissements', 'total_remboursements', 'total_frais_interets',
]

df = pd.read_excel(FICHIER, header=None, skiprows=6, names=colonnes)

# Les montants sont du texte « 1,234,567 » : on retire les virgules puis on convertit
for c in colonnes[1:]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '', regex=False))

print('Dimensions :', df.shape)
display(df.head(3))

### 3.3 Contrôle de cohérence (détection d'anomalies)
Le fichier fournit des colonnes **Total**. Elles doivent être égales à la somme des 3 guichets. On vérifie cette règle comptable.

In [ ]:
ecart_dec  = df.gra_decaissements + df.prgt_decaissements + df.rst_decaissements - df.total_decaissements
ecart_rem  = df.gra_rachats + df.prgt_remboursements + df.rst_remboursements - df.total_remboursements
ecart_frais = df.gra_frais_payes + df.prgt_interets_payes + df.rst_interets_payes - df.total_frais_interets

print('Écart max décaissements  :', ecart_dec.abs().max())
print('Écart max remboursements :', ecart_rem.abs().max())
print('Lignes incohérentes sur frais/intérêts :')
display(df.loc[ecart_frais != 0, ['annee', 'prgt_interets_payes', 'total_frais_interets']].assign(ecart=ecart_frais[ecart_frais != 0]))

**Anomalie détectée :** pour **2010** et **2026**, `prgt_interets_payes` vaut `541.121` et `37.704`. Ce sont en réalité `541 121` et `37 704` DTS : le séparateur de milliers a été interprété comme une virgule décimale lors de l'export. Les totaux du FMI le confirment (l'écart disparaît exactement si on multiplie par 1000). On corrige donc ces deux valeurs **à partir du total officiel** (et non « à l'œil »).

In [ ]:
anomalie = ecart_frais != 0
df.loc[anomalie, 'prgt_interets_payes'] = (
    df.loc[anomalie, 'total_frais_interets']
    - df.loc[anomalie, 'gra_frais_payes']
    - df.loc[anomalie, 'rst_interets_payes']
)

# Tous les montants sont des entiers (DTS) : on passe en int64
df[colonnes[1:]] = df[colonnes[1:]].round().astype('int64')

# Re-vérification
ecart_frais = df.gra_frais_payes + df.prgt_interets_payes + df.rst_interets_payes - df.total_frais_interets
assert ecart_frais.abs().max() == 0
print('✅ Corrigé. Valeurs 2010 et 2026 :')
display(df.loc[df.annee.isin([2010, 2026]), ['annee', 'prgt_interets_payes']])

### 3.4 Création d'une variable catégorielle : la décennie

Le dataset ne contient que des variables numériques. Pour pouvoir faire des `GROUP BY`, on crée la variable catégorielle `decennie` (1980s, 1990s, …).

In [ ]:
df['decennie'] = (df['annee'] // 10 * 10).astype(str) + 's'
print(df['decennie'].value_counts().sort_index())

### 3.5 Exploration avec Pandas

In [ ]:
print('--- 10 premières lignes ---')
display(df.head(10))
print('\n--- Noms et types des colonnes ---')
display(df.dtypes.to_frame('type'))
print('Nombre d\'observations :', len(df))
print('Nombre de colonnes     :', df.shape[1])

In [ ]:
print('--- Valeurs manquantes par colonne ---')
print(df.isna().sum().sum(), 'valeur(s) manquante(s) au total')
print('\n--- Doublons ---')
print('Lignes dupliquées :', df.duplicated().sum(), '| Années dupliquées :', df['annee'].duplicated().sum())
print('\n--- Colonnes constantes ---')
print(df.nunique()[df.nunique() == 1])

In [ ]:
stats = df.drop(columns=['annee', 'decennie']).describe().T[['mean', 'min', '50%', 'max']] / 1e9
stats.columns = ['moyenne', 'minimum', 'médiane', 'maximum']
print('Statistiques descriptives (en milliards de DTS) :')
display(stats.round(2))

**Observations.**
- Aucune valeur manquante ni doublon : 43 années distinctes (1984 → 2026).
- `rst_remboursements` vaut **0 pour toutes les années** : le guichet RST est trop récent pour avoir des remboursements enregistrés. Cette colonne n'apporte aucune information analytique.
- Les `PRGT`/`RST` contiennent beaucoup de zéros (le RST n'existe qu'à partir de 2023 dans le fichier) : zéros réels, pas des valeurs manquantes.
- Moyenne annuelle des décaissements totaux : ≈ 13,3 Md DTS, avec un maximum d'environ 37,0 Md DTS.
- Transformations effectuées avant SQLite : noms de colonnes en `snake_case`, conversion texte → entier, correction de 2 valeurs, ajout de `decennie`.

## Partie 4 — Création de la base SQLite

**Rôle des outils**
- `sqlite3` : module standard de Python qui permet de créer et d'interroger une base SQLite (un simple fichier `.db`, sans serveur).
- `pandas` : sert à charger/manipuler les données en mémoire (DataFrame) et à lire le résultat d'une requête SQL avec `read_sql_query()`.
- `to_sql()` : méthode d'un DataFrame qui écrit ses lignes dans une table SQLite. Ici on crée d'abord la table nous-mêmes (types et clé primaire adaptés), puis on y ajoute les données avec `if_exists='append'`.

In [ ]:
DB = DOSSIER / 'fmi_flux.db'
conn = sqlite3.connect(DB)

conn.execute('DROP TABLE IF EXISTS fmi_flux')
conn.execute("""
CREATE TABLE fmi_flux (
    annee                 INTEGER PRIMARY KEY,   -- une ligne par année
    gra_decaissements     INTEGER NOT NULL,
    gra_rachats           INTEGER NOT NULL,
    gra_frais_payes       INTEGER NOT NULL,
    prgt_decaissements    INTEGER NOT NULL,
    prgt_remboursements   INTEGER NOT NULL,
    prgt_interets_payes   INTEGER NOT NULL,
    rst_decaissements     INTEGER NOT NULL,
    rst_remboursements    INTEGER NOT NULL,
    rst_interets_payes    INTEGER NOT NULL,
    total_decaissements   INTEGER NOT NULL,
    total_remboursements  INTEGER NOT NULL,
    total_frais_interets  INTEGER NOT NULL,
    decennie              TEXT    NOT NULL       -- variable catégorielle créée
)
""")

df.to_sql('fmi_flux', conn, if_exists='append', index=False)
conn.commit()

def sql(requete, afficher=True):
    """Exécute une requête SELECT et renvoie un DataFrame."""
    res = pd.read_sql_query(requete, conn)
    if afficher:
        display(res)
        print(f'→ {len(res)} ligne(s)')
    return res

nb_sql = pd.read_sql_query('SELECT COUNT(*) AS n FROM fmi_flux', conn).loc[0, 'n']
print('Lignes importées :', nb_sql, '| Lignes du DataFrame :', len(df))
assert nb_sql == len(df)
print('✅ Import vérifié. Base créée :', DB.name)

In [ ]:
sql('SELECT annee, decennie, total_decaissements, total_remboursements FROM fmi_flux LIMIT 5;')
sql('PRAGMA table_info(fmi_flux);');

## Partie 5 — Exploitation du dataset avec SQL (15 exercices)

Convention : les montants sont en **DTS**. Pour la lisibilité, on les divise parfois par `1e9` (= milliards de DTS, abrégé « Md »).

### Niveau 1 — Requêtes fondamentales

**Exercice 1 — Afficher toutes les observations.**

In [ ]:
q01 = """
SELECT *
FROM fmi_flux;
"""
r01 = sql(q01)

**Exercice 2 — Sélectionner certaines colonnes** (année, décaissements totaux, remboursements totaux).

In [ ]:
q02 = """
SELECT annee, total_decaissements, total_remboursements
FROM fmi_flux;
"""
r02 = sql(q02)

**Exercice 3 — Limiter le nombre de résultats** : les 10 années les plus récentes.

In [ ]:
q03 = """
SELECT annee, total_decaissements
FROM fmi_flux
ORDER BY annee DESC
LIMIT 10;
"""
r03 = sql(q03)

**Exercice 4 — Trier les observations** : les 5 années où le FMI a le plus décaissé.

In [ ]:
q04 = """
SELECT annee,
       ROUND(total_decaissements / 1e9, 2) AS decaissements_Md_DTS
FROM fmi_flux
ORDER BY total_decaissements DESC
LIMIT 5;
"""
r04 = sql(q04)

**Exercice 5 — Filtrer avec `WHERE`** : années où les décaissements ont dépassé 20 Md DTS.

In [ ]:
q05 = """
SELECT annee,
       ROUND(total_decaissements / 1e9, 2) AS decaissements_Md_DTS
FROM fmi_flux
WHERE total_decaissements > 20e9
ORDER BY annee;
"""
r05 = sql(q05)

### Niveau 2 — Agrégations

**Exercice 6 — Nombre total d'enregistrements.**

In [ ]:
q06 = """
SELECT COUNT(*) AS nb_annees
FROM fmi_flux;
"""
r06 = sql(q06)

**Exercice 7 — Calculer une moyenne** : décaissements et remboursements annuels moyens.

In [ ]:
q07 = """
SELECT ROUND(AVG(total_decaissements)  / 1e9, 2) AS moy_decaissements_Md,
       ROUND(AVG(total_remboursements) / 1e9, 2) AS moy_remboursements_Md
FROM fmi_flux;
"""
r07 = sql(q07)

**Exercice 8 — Valeurs minimale et maximale** des décaissements, et période couverte.

In [ ]:
q08 = """
SELECT ROUND(MIN(total_decaissements) / 1e9, 2) AS min_decaissements_Md,
       ROUND(MAX(total_decaissements) / 1e9, 2) AS max_decaissements_Md,
       MIN(annee) AS premiere_annee,
       MAX(annee) AS derniere_annee
FROM fmi_flux;
"""
r08 = sql(q08)

**Exercice 9 — Regrouper avec `GROUP BY`** : totaux par décennie.

In [ ]:
q09 = """
SELECT decennie,
       COUNT(*) AS nb_annees,
       ROUND(SUM(total_decaissements)  / 1e9, 1) AS decaissements_Md,
       ROUND(SUM(total_remboursements) / 1e9, 1) AS remboursements_Md
FROM fmi_flux
GROUP BY decennie
ORDER BY decennie;
"""
r09 = sql(q09)

**Exercice 10 — Filtrer les groupes avec `HAVING`** : décennies dont le décaissement annuel moyen dépasse 10 Md DTS.

In [ ]:
q10 = """
SELECT decennie,
       ROUND(AVG(total_decaissements) / 1e9, 2) AS moy_decaissements_Md
FROM fmi_flux
GROUP BY decennie
HAVING AVG(total_decaissements) > 10e9
ORDER BY moy_decaissements_Md DESC;
"""
r10 = sql(q10)

### Niveau 3 — Analyses avancées

**Exercice 11 — Combiner `AND` et `OR`** : années où le FMI a beaucoup prêté via le GRA (> 15 Md) **et** via le PRGT (> 2 Md), **ou** où le guichet RST a été utilisé.

In [ ]:
q11 = """
SELECT annee,
       ROUND(gra_decaissements  / 1e9, 1) AS gra_Md,
       ROUND(prgt_decaissements / 1e9, 1) AS prgt_Md,
       ROUND(rst_decaissements  / 1e9, 1) AS rst_Md
FROM fmi_flux
WHERE (gra_decaissements > 15e9 AND prgt_decaissements > 2e9)
   OR rst_decaissements > 0
ORDER BY annee;
"""
r11 = sql(q11)

**Exercice 12 — `BETWEEN` et `IN`** : la période 2008-2012 (crise financière) et les années 2020-2022 (période Covid et suites).

In [ ]:
q12 = """
SELECT annee,
       ROUND(total_decaissements  / 1e9, 1) AS decaissements_Md,
       ROUND(total_remboursements / 1e9, 1) AS remboursements_Md
FROM fmi_flux
WHERE annee BETWEEN 2008 AND 2012
   OR annee IN (2020, 2021, 2022)
ORDER BY annee;
"""
r12 = sql(q12)

**Exercice 13 — Classement** : les 10 années au plus fort **flux net** (décaissements − remboursements), avec `ORDER BY` et la fonction de fenêtre `RANK()`.

In [ ]:
q13 = """
SELECT RANK() OVER (ORDER BY total_decaissements - total_remboursements DESC) AS rang,
       annee,
       ROUND((total_decaissements - total_remboursements) / 1e9, 2) AS flux_net_Md
FROM fmi_flux
ORDER BY rang
LIMIT 10;
"""
r13 = sql(q13)

**Exercice 14 — Sous-requête** : années où les décaissements dépassent la moyenne de toute la période.

In [ ]:
q14 = """
SELECT annee,
       ROUND(total_decaissements / 1e9, 2) AS decaissements_Md
FROM fmi_flux
WHERE total_decaissements > (SELECT AVG(total_decaissements) FROM fmi_flux)
ORDER BY annee;
"""
r14 = sql(q14)

**Exercice 15 — Analyse synthétique (question économique).**
*Question : le FMI a-t-il été prêteur net ou récepteur net de fonds selon les décennies ? Quelle est la part des prêts concessionnels (PRGT + RST) dans ses décaissements, et quel est le poids du coût du crédit (frais + intérêts) dans les sorties des emprunteurs ?*

In [ ]:
q15 = """
SELECT decennie,
       ROUND(SUM(total_decaissements)  / 1e9, 1) AS decaisse_Md,
       ROUND(SUM(total_remboursements) / 1e9, 1) AS rembourse_Md,
       ROUND((SUM(total_decaissements) - SUM(total_remboursements)) / 1e9, 1) AS flux_net_Md,
       ROUND(100.0 * SUM(prgt_decaissements + rst_decaissements)
             / SUM(total_decaissements), 1) AS part_concessionnel_pct,
       ROUND(100.0 * SUM(total_frais_interets)
             / SUM(total_remboursements + total_frais_interets), 1) AS part_cout_credit_pct
FROM fmi_flux
GROUP BY decennie
ORDER BY decennie;
"""
r15 = sql(q15)

## Partie 6 — Analyse et interprétation

On retient **cinq requêtes significatives** : Q4, Q9, Q13, Q14 et Q15.

### 6.1 Résultats sous forme de tableaux Pandas et interprétation

In [ ]:
selection = {
    'Q4  — Top 5 des années de décaissement': r04,
    'Q9  — Totaux par décennie': r09,
    'Q13 — Classement par flux net': r13,
    'Q14 — Années au-dessus de la moyenne': r14,
    'Q15 — Synthèse par décennie': r15,
}
for titre, tab in selection.items():
    print('\n' + titre)
    display(tab)

**Interprétation**

- **Q4 — Pics de décaissement.** Les 5 plus fortes années sont 2020 (36,96 Md DTS), 2011 (34,56), 2022 (27,33), 2002 (26,58) et 2018 (25,45). Deux d'entre elles (2011, 2020) coïncident avec de grandes crises (suites de la crise financière de 2008 et de la zone euro ; pandémie de Covid-19) : le FMI joue un rôle de **prêteur en période de crise**.
- **Q9 — Par décennie.** Les décaissements cumulés passent de 24,1 Md (années 1980, mais seulement 6 années de données 1984-1989) à 163,1 Md (années 2010). Les années 2020 totalisent déjà 151,0 Md en seulement 7 années (2026 partielle) : le **rythme annuel moyen est le plus élevé** de la série (21,6 Md/an contre 16,3 Md/an dans les années 2010, cf. Q10).
- **Q13 — Flux net.** Les 3 plus forts flux nets sont 2020 (+32,35 Md), 2011 (+32,15) et 2009 (+20,81) : ce sont les années où les emprunteurs ont reçu beaucoup plus qu'ils n'ont remboursé, donc des années de **transfert net de ressources** vers les pays membres.
- **Q14 — Au-dessus de la moyenne (13,26 Md).** 19 années sur 43 (44 %) dépassent la moyenne. Elles se regroupent en **trois épisodes** : 1995-2003, 2008-2013 et 2018-2025. Les décaissements sont donc **par vagues**, et non répartis régulièrement.
- **Q15 — Synthèse.** Le FMI est **prêteur net** dans les années 1990 (+32,8 Md), 2010 (+31,2) et 2020 (+49,6), mais **récepteur net** dans les années 1980 (−10,0) et 2000 (−15,2) : les années 2000 correspondent à de gros remboursements (par exemple 30,1 Md en 2005). La **part concessionnelle** (PRGT + RST) reste faible (5 à 8 %) jusqu'aux années 2010 puis **bondit à 21,0 %** dans les années 2020. Enfin, le **coût du crédit** pèse 21,2 % des sorties des emprunteurs dans les années 2020, contre 10-11 % dans les années 2000-2010.

### 6.2 Graphiques (Matplotlib, alimentés par des requêtes SQL)

In [ ]:
# Graphique 1 : décaissements vs remboursements par année (en Md DTS)
g1 = sql("""
SELECT annee,
       total_decaissements  / 1e9 AS decaissements,
       total_remboursements / 1e9 AS remboursements
FROM fmi_flux ORDER BY annee;
""", afficher=False)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(g1.annee, g1.decaissements, marker='o', ms=4, label='Décaissements')
ax.plot(g1.annee, g1.remboursements, marker='s', ms=4, label='Remboursements')
ax.set_title('FMI : décaissements et remboursements annuels (1984-2026)')
ax.set_xlabel('Année (2026 : données jusqu\'au 30 sept.)')
ax.set_ylabel('Milliards de DTS')
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(DOSSIER / 'graphique1_flux_annuels.png', dpi=150); plt.show()

**Lecture :** les deux courbes sont très volatiles. Les décaissements culminent en 2011 et 2020 ; les remboursements ont leur pic en 2005 (30,1 Md), puis en 2014-2015 ; ils dépassent les décaissements notamment autour de 2005-2007 et en 2015-2017.

In [ ]:
# Graphique 2 : flux net par décennie (en Md DTS)
g2 = sql("""
SELECT decennie,
       (SUM(total_decaissements) - SUM(total_remboursements)) / 1e9 AS flux_net
FROM fmi_flux GROUP BY decennie ORDER BY decennie;
""", afficher=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
couleurs = ['tab:red' if v < 0 else 'tab:green' for v in g2.flux_net]
barres = ax.bar(g2.decennie, g2.flux_net, color=couleurs)
ax.bar_label(barres, fmt='%.1f', padding=3)
ax.axhline(0, color='black', lw=.8)
ax.set_title('Flux net du FMI par décennie (décaissements − remboursements)')
ax.set_ylabel('Milliards de DTS')
plt.tight_layout(); plt.savefig(DOSSIER / 'graphique2_flux_net_decennie.png', dpi=150); plt.show()

**Lecture :** alternance de décennies de transfert net vers les emprunteurs (vert : 1990s, 2010s, 2020s) et de décennies de remboursement net (rouge : 1980s, 2000s). Attention : les années 1980 ne contiennent que 6 années (1984-1989) et les années 2020 que 7 années.

In [ ]:
# Graphique 3 : décaissements par guichet depuis 2000 (barres empilées)
g3 = sql("""
SELECT annee,
       gra_decaissements  / 1e9 AS GRA,
       prgt_decaissements / 1e9 AS PRGT,
       rst_decaissements  / 1e9 AS RST
FROM fmi_flux WHERE annee >= 2000 ORDER BY annee;
""", afficher=False)

ax = g3.set_index('annee').plot(kind='bar', stacked=True, figsize=(11, 4.5), width=.8)
ax.set_title('Décaissements du FMI par guichet (depuis 2000)')
ax.set_xlabel('Année'); ax.set_ylabel('Milliards de DTS')
ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(DOSSIER / 'graphique3_guichets.png', dpi=150); plt.show()

**Lecture :** le GRA (prêts aux conditions du marché) domine presque toutes les années ; le PRGT est visible en 2020 (6,8 Md) ; le RST n'apparaît qu'à partir de 2023.

### 6.3 Trois conclusions

1. **Le FMI intervient par vagues, en période de crise.** Les décaissements sont très irréguliers (de 1,3 à 37,0 Md DTS par an) ; 19 années sur 43 dépassent la moyenne de 13,3 Md, regroupées en trois épisodes (1995-2003, 2008-2013, 2018-2025) avec des pics en 2011 et 2020.
2. **Le rôle de prêteur net n'est pas permanent.** Le flux net est positif dans les années 1990, 2010 et 2020, mais négatif dans les années 1980 et 2000, quand les remboursements dépassent les nouveaux prêts (pic de 30,1 Md remboursés en 2005).
3. **Les années 2020 marquent un changement de structure.** Le rythme annuel de prêt est le plus élevé de la série (21,6 Md/an), la part des guichets concessionnels (PRGT + RST) passe de 5,3 % à 21,0 %, et le coût du crédit (frais + intérêts) atteint un record (6,03 Md en 2024) avec 21,2 % des sorties des emprunteurs.

### Limites de l'analyse
- Série **agrégée** de tous les pays membres : on ne peut pas identifier quels pays expliquent les pics.
- Montants **nominaux** en DTS (non corrigés de l'inflation) : la croissance apparente est en partie nominale.
- **2026 est partielle** (jusqu'au 30/09) et les décennies ont des durées inégales (6, 10, 10, 10, 7 années) : comparer les moyennes annuelles plutôt que les cumuls.
- Les rapprochements avec des événements (Covid, crise de 2008) sont des **coïncidences temporelles**, pas une preuve de causalité.

## Partie 7 — Livrables : export des requêtes et vérification

In [ ]:
titres = {
    1: 'Toutes les observations', 2: 'Sélection de colonnes', 3: 'Limiter les résultats',
    4: 'Tri : top 5 des décaissements', 5: 'Filtre WHERE', 6: 'COUNT', 7: 'AVG',
    8: 'MIN / MAX', 9: 'GROUP BY décennie', 10: 'HAVING', 11: 'AND / OR',
    12: 'BETWEEN / IN', 13: 'Classement RANK()', 14: 'Sous-requête', 15: 'Analyse synthétique',
}
lignes = ['-- TP SQL — Flux financiers du FMI | base : fmi_flux.db | table : fmi_flux', '']
for i, t in titres.items():
    lignes += [f'-- Exercice {i} : {t}', globals()[f'q{i:02d}'].strip(), '']
(DOSSIER / 'requetes_sql.sql').write_text('\n'.join(lignes), encoding='utf-8')

conn.close()
print('✅ Fichiers présents dans', DOSSIER)
for f in sorted(DOSSIER.iterdir()):
    if f.suffix in ('.db', '.sql', '.png', '.xlsx', '.ipynb'):
        print(' -', f.name)

### Livrables à remettre
1. ✅ Ce notebook (`.ipynb`) · 2. ✅ Dataset original `extrep1.xls.xlsx` · 3. ✅ Base SQLite `fmi_flux.db` · 4. ✅ Les 15 requêtes et leurs résultats (ci-dessus + `requetes_sql.sql`) · 5. ✅ Synthèse d'une page (document joint).